In [2]:
import pandas as pd
import sqlite3

df = pd.read_csv('DataCoSupplyChainDatasetRefined.csv')

conn = sqlite3.connect('supply_chain.db')
df.to_sql('orders', conn, index=False, if_exists='replace')

18891

In [4]:
print(df.columns.tolist())

['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude_src', 'longitude_src', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date_dateorders', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_image', 'product_name', 'product_price', 'product_status', 'shipping_date_dateorders', 'shipping_mode', 

In [5]:
query = """
WITH Shipping_Base_Logic AS (
    SELECT
        *,

        (days_for_shipping_real - days_for_shipment_scheduled) AS delay_days,

        CASE WHEN delivery_status = 'Late delivery' THEN 1 ELSE 0 END AS is_late_flag
    FROM orders
),

Refined_Data AS (
    SELECT
        *,
        CASE
            WHEN delay_days > 2 THEN 'High Risk'
            WHEN delay_days BETWEEN 1 AND 2 THEN 'Medium Risk'
            WHEN delay_days = 0 THEN 'On Time'
            ELSE 'Advance Shipping'
        END AS shipping_priority_level,
        ROUND((benefit_per_order / NULLIF(sales_per_customer, 0)), 4) AS profit_margin_ratio
    FROM Shipping_Base_Logic
)

SELECT
    category_name,
    product_name,
    order_city_en,
    order_country_en,
    shipping_mode,
    delivery_status,
    shipping_priority_level,
    delay_days,
    is_late_flag,
    sales AS total_sales,
    benefit_per_order AS net_profit,
    latitude_dest,
    longitude_dest
FROM Refined_Data
ORDER BY delay_days DESC;
"""

result_df = pd.read_sql_query(query, conn)
result_df.head(10)

,category_name,product_name,order_city_en,order_country_en,shipping_mode,delivery_status,shipping_priority_level,delay_days,is_late_flag,total_sales,net_profit,latitude_dest,longitude_dest
0,Sporting Goods,Smart watch,Tokyo,Japan,Second Class,Shipping canceled,High Risk,4,0,327.750000,130.580002,35.680400,139.769017
1,Sporting Goods,Smart watch,Mandurah,Australia,Second Class,Late delivery,High Risk,4,1,327.750000,131.169998,-32.536104,115.742408
2,Women's Apparel,Nike Men's Dri-FIT Victory Golf Polo,Murray Bridge,Australia,Second Class,Late delivery,High Risk,4,1,100.000000,33.599998,-35.120051,139.273696
3,Women's Apparel,Nike Men's Dri-FIT Victory Golf Polo,Raipur,India,Second Class,Late delivery,High Risk,4,1,100.000000,6.150000,26.041164,74.019730
4,Cleats,Perfect Fitness Perfect Rip Deck,San Francisco,USA,Second Class,Late delivery,High Risk,4,1,119.980003,30.570000,37.774929,-122.419415
5,Cardio Equipment,Nike Men's Free 5.0+ Running Shoe,Nantes,France,Second Class,Late delivery,High Risk,4,1,299.970001,76.040001,47.218371,-1.553621
6,Trade-In,Glove It Women's Mod Oval 3-Zip Carry All Gol,Nice,France,Second Class,Late delivery,High Risk,4,1,65.970001,7.760000,43.710173,7.261953
7,Cardio Equipment,Nike Men's Free 5.0+ Running Shoe,Petapa,Guatemala,Second Class,Late delivery,High Risk,4,1,299.970001,101.389999,14.581308,-90.545982
8,Women's Apparel,Nike Men's Dri-FIT Victory Golf Polo,Mexico City,Mexico,Second Class,Late delivery,High Risk,4,1,150.000000,43.660000,19.432608,-99.133208
9,Cardio Equipment,Nike Men's Free 5.0+ Running Shoe,Kuito,Angola,Second Class,Late delivery,High Risk,4,1,99.989998,40.020000,-12.394041,16.941763


In [6]:
query_advanced = """
SELECT
    order_country_en,
    category_name,
    COUNT(*) as total_orders,
    AVG(delay_days) as avg_delay,

    ROUND(100 - (AVG(late_delivery_risk) * 50 + AVG(delay_days) * 10), 2) as resilience_score
FROM (
    SELECT *, (days_for_shipping_real - days_for_shipment_scheduled) AS delay_days
    FROM orders
)
GROUP BY 1, 2
HAVING total_orders > 5
ORDER BY resilience_score ASC;
"""
resilience_df = pd.read_sql_query(query_advanced, conn)
resilience_df.head()

,order_country_en,category_name,total_orders,avg_delay,resilience_score
0,Germany,Consumer Electronics,8,2.500000,25.00
1,Canada,Camping & Hiking,6,2.000000,30.00
2,Argentina,Fishing,6,1.833333,31.67
3,Norway,Women's Apparel,6,2.666667,31.67
4,India,Women's Clothing,9,1.777778,32.22


In [8]:
final_export = result_df.merge(resilience_df[['order_country_en', 'category_name', 'resilience_score']],
                             on=['order_country_en', 'category_name'],
                             how='left')

final_export.to_csv('Master_SupplyChain_Project.csv', index=False)
